# 06_Decision_Tree_Scratch.ipynb

# Decision Tree Classifier (From Scratch)

This notebook implements a Decision Tree Classifier from scratch using the CART algorithm with Gini Impurity.

The implementation includes:
- Gini Impurity
- Best Split Search
- Recursive Tree Building
- Prediction
- Model Evaluation

No machine learning algorithms from sklearn are used.

In [1]:
import numpy as np
import pandas as pd
import pickle
from collections import Counter

In [2]:
# Load prepared datasets

X_train = pd.read_csv("../data/processed/X_train_scaled.csv").values
X_test = pd.read_csv("../data/processed/X_test_scaled.csv").values

y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)
print("Training Labels  :", y_train.shape)
print("Testing Labels   :", y_test.shape)

Training Features: (1760, 7)
Testing Features : (440, 7)
Training Labels  : (1760,)
Testing Labels   : (440,)


In [6]:
#Gini Impurity
def gini_impurity(y):
    """
    Calculate the Gini Impurity of a target array.
    """

    if len(y) == 0:
        return 0

    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)

    gini = 1 - np.sum(probabilities ** 2)

    return gini

In [8]:
#Split Dataset
def split_dataset(X, y, feature_index, threshold):
    """
    Split the dataset based on a feature threshold.
    """

    left_mask = X[:, feature_index] <= threshold
    right_mask = X[:, feature_index] > threshold

    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right = X[right_mask]
    y_right = y[right_mask]

    return X_left, X_right, y_left, y_right

In [9]:
#Best Split Search
def best_split(X, y):
    """
    Find the best feature and threshold that minimize Gini Impurity.
    """

    n_samples, n_features = X.shape

    best_feature = None
    best_threshold = None
    best_gini = float("inf")

    for feature_index in range(n_features):

        thresholds = np.unique(X[:, feature_index])

        for threshold in thresholds:

            X_left, X_right, y_left, y_right = split_dataset(
                X, y, feature_index, threshold
            )

            # Ignore invalid splits
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            left_gini = gini_impurity(y_left)
            right_gini = gini_impurity(y_right)

            weighted_gini = (
                (len(y_left) / n_samples) * left_gini
                + (len(y_right) / n_samples) * right_gini
            )

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature_index
                best_threshold = threshold

    return best_feature, best_threshold

In [10]:
#Majority Class
def majority_class(y):
    """
    Return the most common class in the target array.
    """

    return Counter(y).most_common(1)[0][0]

In [11]:
#Build Decision Tree
def build_tree(X, y, depth=0, max_depth=10):
    """
    Recursively build the Decision Tree.
    """

    # If all samples belong to one class
    if len(np.unique(y)) == 1:
        return y[0]

    # Stop if maximum depth is reached
    if depth >= max_depth:
        return majority_class(y)

    feature, threshold = best_split(X, y)

    # No valid split found
    if feature is None:
        return majority_class(y)

    X_left, X_right, y_left, y_right = split_dataset(
        X, y, feature, threshold
    )

    # Safety check
    if len(y_left) == 0 or len(y_right) == 0:
        return majority_class(y)

    left_tree = build_tree(
        X_left,
        y_left,
        depth + 1,
        max_depth
    )

    right_tree = build_tree(
        X_right,
        y_right,
        depth + 1,
        max_depth
    )

    return {
        "feature": feature,
        "threshold": threshold,
        "left": left_tree,
        "right": right_tree,
    }

In [13]:
#Prediction for a Single Sample
def predict_sample(tree, sample):
    """
    Predict the class label for a single sample.
    """

    # Leaf node
    if not isinstance(tree, dict):
        return tree

    feature = tree["feature"]
    threshold = tree["threshold"]

    if sample[feature] <= threshold:
        return predict_sample(tree["left"], sample)
    else:
        return predict_sample(tree["right"], sample)

In [14]:
#Prediction for Multiple Samples
def predict(tree, X):
    """
    Predict class labels for multiple samples.
    """

    predictions = [predict_sample(tree, sample) for sample in X]

    return np.array(predictions)

In [15]:
# Build the Decision Tree

decision_tree = build_tree(
    X_train,
    y_train,
    max_depth=10
)

print("Decision Tree training completed.")

Decision Tree training completed.


In [16]:
# Make predictions

y_pred = predict(decision_tree, X_test)

print("First 10 Predictions :", y_pred[:10])
print("First 10 Actual Labels:", y_test[:10])

First 10 Predictions : [16  1  6 11 16  3 20  2  1 16]
First 10 Actual Labels: [16  1  6 11 16  3 20  2  1 16]


In [17]:
def accuracy_score(y_true, y_pred):
    """
    Calculate classification accuracy.
    """

    return np.mean(y_true == y_pred)

In [18]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9182


In [20]:
# Save the trained Decision Tree

with open("../saved_models/decision_tree_model.pkl", "wb") as file:
    pickle.dump(decision_tree, file)

print("Decision Tree model saved successfully.")

Decision Tree model saved successfully.


## Conclusion

A Decision Tree Classifier was successfully implemented from scratch using the CART algorithm with Gini Impurity.

The model was trained, evaluated, and saved for future use without relying on any built-in machine learning algorithms.